# ðŸŽ¬ AI Influencer Engine
**Self-Contained Colab | Apache 2.0 | Commercial Safe**

Run cells 1â†’7 to produce your AI influencer image,
then upload `stage3_synced.jpg` to **Kling AI / Pika** for animation.

---

## How It Works
| Stage | Model | What Happens | Time on T4 |
|-------|-------|-------------|------------|
| **1** | Realistic Vision V5.1 + IP-Adapter | Generates photorealistic portrait matching your reference face | ~3 min |
| **2** | ffmpeg | Extracts best pose frame from your reference reel | ~5 sec |
| **3** | SD 1.5 ControlNet OpenPose + IP-Adapter | Places influencer face on reference body pose | ~3 min |

**Total: ~6-8 minutes | Output: `stage3_synced.jpg` â€” upload to Kling AI for animation**

---

## License â€” All Commercial Safe
| Model | License |
|-------|---------|
| Realistic Vision V5.1 | CreativeML OpenRAIL+M (commercial OK) |
| IP-Adapter | Apache 2.0 |
| ControlNet OpenPose | Apache 2.0 |

No InsightFace. No ArcFace. No FaceID. No FLUX dev. No InstantID.

---

> **BEFORE YOU START:**
> 1. Runtime â†’ Change runtime type â†’ **T4 GPU** â†’ Save
> 2. (Optional) Add `GEMINI_API_KEY` in Colab Secrets for auto outfit detection
> 3. Run cells top to bottom, one at a time

---
## 🖥️ Cell 1 — GPU Check
Run this first. Confirms you have enough VRAM before wasting time on downloads.

| GPU | VRAM | Tier | Speed |
|-----|------|------|-------|
| T4 | 15 GB | LOW | ~18 min/video |
| L4 | 24 GB | MEDIUM | ~10 min/video |
| A100 | 40 GB | HIGH | ~5 min/video |


In [ ]:
import torch

print('=' * 60)
print('GPU STATUS')
print('=' * 60)

if not torch.cuda.is_available():
    raise EnvironmentError(
        'No GPU detected!\n'
        'Fix: Runtime > Change runtime type > T4 GPU > Save'
    )

GPU_NAME   = torch.cuda.get_device_name(0)
VRAM_TOTAL = torch.cuda.get_device_properties(0).total_memory / 1024**3
VRAM_FREE  = (torch.cuda.get_device_properties(0).total_memory
              - torch.cuda.memory_allocated()) / 1024**3

print(f'  GPU        : {GPU_NAME}')
print(f'  VRAM Total : {VRAM_TOTAL:.1f} GB')
print(f'  VRAM Free  : {VRAM_FREE:.1f} GB')
print(f'  PyTorch    : {torch.__version__}')
print(f'  CUDA       : {torch.version.cuda}')
print()

if VRAM_TOTAL < 6:
    raise EnvironmentError(
        f'Only {VRAM_TOTAL:.1f} GB VRAM. Need >= 6 GB.\n'
        'Switch to T4: Runtime > Change runtime type > T4 GPU.'
    )
elif VRAM_TOTAL < 14:
    TIER = 'LOW'
    print('Tier  : LOW (T4, ~15 GB)')
    print('Models: Wan2.1-1.3B Lite + SDXL FP16')
    print('Speed : ~18-20 min/video')
elif VRAM_TOTAL < 26:
    TIER = 'MEDIUM'
    print('Tier  : MEDIUM (L4, ~24 GB)')
    print('Models: Wan2.1-1.3B + SDXL Full')
    print('Speed : ~10-12 min/video')
else:
    TIER = 'HIGH'
    print('Tier  : HIGH (A100, 40+ GB)')
    print('Models: Wan2.1-14B + SDXL Full')
    print('Speed : ~5-6 min/video')

print()
print('GPU check passed. Run Cell 2 next.')


---
## 📦 Cell 2 — Install Dependencies
Installs all required libraries. **Runs once per Colab session (~2-4 minutes).**
If you restart the runtime, run this cell again before running the pipeline.

All packages are open-source. No MediaPipe — we use OpenCV for face detection.


In [ ]:
import subprocess, sys

print('Step 1/3 — System packages (ffmpeg, libgl)...')
subprocess.run(['apt-get', 'install', '-y', 'ffmpeg', 'libgl1-mesa-glx'], capture_output=True)
print('  ffmpeg: OK')

print()
print('Step 2/3 — Python packages...')
r = subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'diffusers>=0.28.0',
    'transformers>=4.40.0',
    'accelerate>=0.30.0',
    'controlnet-aux>=0.0.9',
    'huggingface-hub>=0.22.0',
    'safetensors>=0.4.0',
    'sentencepiece',
    'imageio[ffmpeg]',
    'opencv-python-headless',
    'google-generativeai',
], capture_output=True, text=True)

if r.returncode == 0:
    print('  All Python packages: OK')
else:
    print('  Warning:', r.stderr[-300:])

print()
print('Step 3/3 — Verification...')
import importlib, torch
for mod in ['torch', 'diffusers', 'transformers', 'cv2', 'imageio']:
    try:
        m = importlib.import_module(mod)
        v = getattr(m, '__version__', 'OK')
        print(f'  {mod:<15}: {v}')
    except ImportError:
        print(f'  {mod:<15}: MISSING — re-run this cell')

print()
print('All dependencies ready. Run Cell 3 next.')


---
## 📤 Cell 3 — Provide Your Input Files

Choose **either Option A (upload from your computer)** or **Option B (use files stored in Google Drive)**.

If you have already uploaded `face.png` and `Gawriee.mp4` to your `AMTCE` folder in Google Drive, set `USE_DRIVE_FILES = True`.

You need exactly **two files**:

### File 1: Reference Face Image
- A clear photo of the face you want your AI influencer to have
- **Best results:** front-facing, eyes open, good lighting, no sunglasses
- Source: AI-generated face (ChatGPT, Midjourney), Pinterest photo, or your own photo
- Format: JPG or PNG

### File 2: Reference Video (Instagram Reel)
- A trending reel whose body movement and pose you want to replicate
- The AI extracts only the **body pose skeleton** — it does NOT copy the person's identity
- Format: MP4, 5-30 seconds works best



In [ ]:
from google.colab import files, drive
from IPython.display import display, Image as IPImage
import os, shutil

# ================================================================
# ⚙️ Set this to True to use files already in your Google Drive
# Set this to False to manually upload files from your computer
# ================================================================
USE_DRIVE_FILES = True

REFERENCE_FACE = '/content/input_face.jpg'
REFERENCE_VIDEO = '/content/input_video.mp4'

if USE_DRIVE_FILES:
    print('Using Option B: Files stored in Google Drive')
    print('=' * 60)
    try:
        drive.mount('/content/drive', force_remount=False)
    except:
        print('Warning: Drive already mounted or error mounting.')

    drive_face_path = '/content/drive/MyDrive/AMTCE/face.png'
    drive_video_path = '/content/drive/MyDrive/AMTCE/Gawriee.mp4'

    if not os.path.exists(drive_face_path):
         raise FileNotFoundError(f"Face image not found in Drive at: {drive_face_path}")
    if not os.path.exists(drive_video_path):
         raise FileNotFoundError(f"Video not found in Drive at: {drive_video_path}")

    # Copy from Drive to /content to prevent path/filename issues
    shutil.copy(drive_face_path, REFERENCE_FACE)
    shutil.copy(drive_video_path, REFERENCE_VIDEO)
    print(f'✅ Copied Face: {drive_face_path}')
    print(f'✅ Copied Video: {drive_video_path}')
else:
    print('Using Option A: Upload manually')
    print('=' * 60)
    print('UPLOAD 1/2 — FACE IMAGE (JPG or PNG)')
    print('Tip: clear front-facing photo, eyes open, good lighting')
    print('=' * 60)
    face_upload = files.upload()
    if not face_upload:
        raise ValueError('No file uploaded. Please upload a face image.')

    original_face = list(face_upload.keys())[0]
    with open(REFERENCE_FACE, 'wb') as f:
        f.write(face_upload[original_face])
    print(f'✅ Face uploaded and saved as /content/input_face.jpg')

    print()
    print('=' * 60)
    print('UPLOAD 2/2 — REFERENCE VIDEO (MP4)')
    print('Tip: 5-15 second trending Instagram reel works best')
    print('=' * 60)
    video_upload = files.upload()
    if not video_upload:
        raise ValueError('No file uploaded. Please upload a reference video.')

    original_video = list(video_upload.keys())[0]
    with open(REFERENCE_VIDEO, 'wb') as f:
        f.write(video_upload[original_video])
    print(f'✅ Video uploaded and saved as /content/input_video.mp4')

print()
print('Validation:')
for label, path in [('Face', REFERENCE_FACE), ('Video', REFERENCE_VIDEO)]:
    exists = os.path.exists(path)
    size   = os.path.getsize(path) / 1024**2
    status = 'OK' if exists and size > 0.01 else 'ERROR'
    print(f'  {label:<6}: {status}  ({size:.2f} MB)')

print()
print('Preview of your reference face:')
display(IPImage(REFERENCE_FACE, width=300))


import os, subprocess

# ================================================================
# STEP 1: Scene / mood description (outfit is auto-detected below)
#   Edit this to match the BACKGROUND and MOOD you want.
# ================================================================
STYLE_SCENE = (
    'rooftop with city skyline at golden hour, confident and graceful'
)

# ================================================================
# OUTPUT DIRECTORY
# ================================================================
OUTPUT_DIR = '/content/drive/MyDrive/AMTCE/Influencer_Output'
# OUTPUT_DIR = '/content/Influencer_Output'  # use this if Drive fails

# TECHNICAL SETTINGS
IP_ADAPTER_SCALE = 0.65   # 0.5 = subtle identity, 0.8 = strong identity
CONTROLNET_SCALE = 0.85   # 0.7 = loose pose, 1.0 = strict pose match
SEED             = 42     # change for variation

REFERENCE_FACE  = '/content/input_face.jpg'
REFERENCE_VIDEO = '/content/input_video.mp4'
FRAME_PATH      = '/content/stage2_ref_frame.jpg'

# Mount Drive
if OUTPUT_DIR.startswith('/content/drive'):
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        print('Google Drive mounted.')
    except Exception as e:
        print(f'Drive mount failed: {e}')
        OUTPUT_DIR = '/content/Influencer_Output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ================================================================
# STEP 2: Extract reference frame EARLY â€” we need it for outfit detection
# ================================================================
if os.path.exists(REFERENCE_VIDEO):
    print('Extracting reference frame for outfit detection...')
    r = subprocess.run([
        'ffmpeg', '-y', '-i', REFERENCE_VIDEO,
        '-vf', 'thumbnail=300,scale=512:-1',
        '-frames:v', '1', '-q:v', '2', FRAME_PATH
    ], capture_output=True)
    exists = os.path.exists(FRAME_PATH)
    print(f'  Frame extracted: {FRAME_PATH}' if exists else '  Frame extraction failed')
else:
    print(f'No reference video at {REFERENCE_VIDEO} â€” run Cell 3 first.')

# ================================================================
# STEP 3: Auto-detect outfit colour + style from the frame
#   Priority: Gemini Vision (accurate) â†’ OpenCV colour (fallback)
# ================================================================
outfit_desc = 'silk saree with traditional jewellery'  # default

if os.path.exists(FRAME_PATH):
    # Try Gemini Vision
    GEMINI_API_KEY = os.getenv('GEMINI_API_KEY', '')
    if not GEMINI_API_KEY:
        try:
            from google.colab import userdata
            GEMINI_API_KEY = userdata.get('GEMINI_API_KEY') or ''
        except: pass

    if GEMINI_API_KEY:
        try:
            import google.generativeai as genai
            from PIL import Image as _PIL
            genai.configure(api_key=GEMINI_API_KEY)
            model_g = genai.GenerativeModel('gemini-1.5-flash')
            frame_pil = _PIL.open(FRAME_PATH).convert('RGB')
            response = model_g.generate_content([
                frame_pil,
                'Describe the outfit worn by the person for a Stable Diffusion prompt. '
                'Include: exact garment type (saree/lehenga/kurta/etc), precise colour, '
                'fabric, pattern, embroidery, jewellery. Under 35 words. '
                'Example: vibrant peacock blue Kanjeevaram silk saree with gold zari border, '
                'temple jewellery, floral embroidery blouse'
            ])
            outfit_desc = response.text.strip().replace('\n', ' ')
            print(f'Gemini Vision outfit detected:')
            print(f'  {outfit_desc}')
        except Exception as e:
            print(f'Gemini detection failed ({e}). Using OpenCV colour fallback...')
    else:
        print('No GEMINI_API_KEY â€” using OpenCV colour analysis (add key to Colab Secrets for better results)')

    # OpenCV colour fallback (runs whenever Gemini is unavailable or fails)
    if outfit_desc == 'silk saree with traditional jewellery':
        try:
            import cv2, numpy as np
            frame_bgr = cv2.imread(FRAME_PATH)
            # Sample the centre 50% of the image (where the outfit usually is)
            h, w = frame_bgr.shape[:2]
            roi = frame_bgr[h//4:3*h//4, w//4:3*w//4]
            hsv = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)
            h_mean = np.mean(hsv[:,:,0])
            s_mean = np.mean(hsv[:,:,1])
            if s_mean < 40:   colour = 'ivory white'
            elif h_mean < 15 or h_mean > 165: colour = 'deep red'
            elif h_mean < 35: colour = 'golden yellow'
            elif h_mean < 75: colour = 'vibrant green'
            elif h_mean < 130: colour = 'royal blue'
            elif h_mean < 155: colour = 'royal purple'
            else: colour = 'rose pink'
            outfit_desc = f'{colour} traditional Indian saree with matching jewellery'
            print(f'OpenCV colour-detected outfit: {outfit_desc}')
        except Exception as cv_e:
            print(f'OpenCV fallback also failed: {cv_e}. Using default.')
else:
    print('No reference frame â€” using default outfit. Run Cell 3 to upload video.')

# ================================================================
# STEP 4: Build final INFLUENCER_CONCEPT
# ================================================================
INFLUENCER_CONCEPT = (
    f'beautiful indian fashion influencer wearing {outfit_desc}, '
    f'{STYLE_SCENE}'
)

print()
print('=' * 60)
print('CONFIGURATION SUMMARY')
print('=' * 60)
preview = INFLUENCER_CONCEPT[:100] + ('...' if len(INFLUENCER_CONCEPT) > 100 else '')
print(f'  Outfit (auto) : {outfit_desc}')
print(f'  Full concept  : {preview}')
print(f'  Output dir    : {OUTPUT_DIR}')
print(f'  IP scale      : {IP_ADAPTER_SCALE}')
print(f'  ControlNet    : {CONTROLNET_SCALE}')
print(f'  Seed          : {SEED}')
print('=' * 60)
print()
print('Configuration ready. Run Cell 5 (Stage 1 - Portrait).')

In [ ]:
import os

# ================================================================
# EDIT THIS — describe your influencer in plain English
# ================================================================
INFLUENCER_CONCEPT = (
    'beautiful indian fashion influencer in a silk saree with gold jewellery, '
    'rooftop with city view at golden hour, confident and graceful'
)

# ================================================================
# OUTPUT DIRECTORY
# Option A (recommended): Save to Google Drive — persists after session
OUTPUT_DIR = '/content/drive/MyDrive/AMTCE/Influencer_Output'
# Option B: Save locally in Colab only (lost when session ends)
# OUTPUT_DIR = '/content/Influencer_Output'
# ================================================================

# TECHNICAL SETTINGS — defaults work well, change only if needed
# Face identity strength: 0.5 = subtle, 0.65 = balanced, 0.8 = strong
IP_ADAPTER_SCALE = 0.65
# Pose match strength: 0.7 = loose, 0.85 = balanced, 1.0 = strict
CONTROLNET_SCALE = 0.85
# Seed: same seed + same inputs = same result. Change for variation.
SEED = 42
# Frames: 81 = ~4 sec at 24fps (safe for T4). Increase on better GPU.
VIDEO_FRAMES = 81

# Mount Drive if using Drive output
if OUTPUT_DIR.startswith('/content/drive'):
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        print('Google Drive mounted.')
    except Exception as e:
        print(f'Drive mount failed: {e}')
        OUTPUT_DIR = '/content/Influencer_Output'
        print(f'Switched to local output: {OUTPUT_DIR}')

os.makedirs(OUTPUT_DIR, exist_ok=True)

print()
print('=' * 60)
print('CONFIGURATION SUMMARY')
print('=' * 60)
preview = INFLUENCER_CONCEPT[:80] + ('...' if len(INFLUENCER_CONCEPT) > 80 else '')
print(f'  Concept      : {preview}')
print(f'  Output dir   : {OUTPUT_DIR}')
print(f'  IP scale     : {IP_ADAPTER_SCALE}  (face identity strength)')
print(f'  ControlNet   : {CONTROLNET_SCALE}  (pose match strength)')
print(f'  Seed         : {SEED}')
print(f'  Video frames : {VIDEO_FRAMES}  (~{VIDEO_FRAMES/24:.1f}s at 24fps)')
print('=' * 60)
print()
print('Configuration ready. Run Cell 5 next (Stage 1 - Portrait).')


---
## 🎨 Cell 5 — Stage 1: Generate AI Influencer Portrait
**Time: ~3-4 minutes on T4**

What this cell does:
1. Detects and crops the face from your reference image using **OpenCV** (no mediapipe — avoids Colab env bugs)
2. Loads **SDXL Base 1.0** — the image generation model
3. Loads **IP-Adapter Plus Face** (CLIP-based) — forces SDXL to match your face
4. Generates a photorealistic portrait at **896x1152px** (Instagram 3:4 ratio)

The output will look like a completely new AI-generated person whose face matches your reference.


In [ ]:
import os, gc, cv2, torch
from PIL import Image
from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler

os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

PORTRAIT_PATH = '/content/stage1_portrait.jpg'

# SD 1.5 native resolution is 512×768 (portrait 2:3).
# Realistic Vision V5.1 generates virtually identical quality to SDXL
# for faces/fashion, at 3× less VRAM. No one can tell the difference
# in a 1080p Instagram crop.
PORTRAIT_PROMPT = (
    f'RAW photo, portrait of a {INFLUENCER_CONCEPT}, '
    '(high detailed skin:1.2), 8k uhd, dslr, soft lighting, '
    'high quality, film grain, Fujifilm XT3, natural skin texture'
)
PORTRAIT_NEG = (
    '(deformed iris, deformed pupils, semi-realistic, cgi, 3d, render, '
    'sketch, cartoon, drawing, anime), text, cropped, out of frame, '
    'worst quality, low quality, jpeg artifacts, ugly, duplicate, '
    'morbid, mutilated, extra fingers, mutated hands, poorly drawn hands, '
    'poorly drawn face, mutation, deformed, blurry, dehydrated, bad anatomy, '
    'bad proportions, extra limbs, cloned face, disfigured, gross proportions'
)

print('STAGE 1 — Portrait Generation (Realistic Vision V5.1 + SD 1.5 IP-Adapter)')
print(f'Peak VRAM target: ~2.5 GB (vs 11 GB with SDXL)')
print()

device = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype  = torch.float16

# Face crop (OpenCV)
def crop_face(path, margin=0.15):
    img = cv2.imread(path)
    h, w = img.shape[:2]
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    det  = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
    faces = det.detectMultiScale(gray, 1.05, 3, minSize=(50,50))
    rgb   = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    if len(faces) > 0:
        x, y, fw, fh = faces[0]
        x1,y1 = max(0,int(x-margin*fw)), max(0,int(y-margin*fh))
        x2,y2 = min(w,int(x+fw+margin*fw)), min(h,int(y+fh+margin*fh))
        print(f'  Face: {x2-x1}x{y2-y1}px')
        return Image.fromarray(rgb[y1:y2, x1:x2])
    s = min(w,h); cx,cy = w//2, h//2
    return Image.fromarray(rgb[cy-s//2:cy+s//2, cx-s//2:cx+s//2])

face_input = crop_face(REFERENCE_FACE).resize((224, 224), Image.LANCZOS)
print(f'  Face ready for IP-Adapter')
print()

# ── Load Realistic Vision (SD 1.5 fine-tune, ~2 GB FP16) ──────────────
print('Loading Realistic Vision V5.1 (SD 1.5, ~2 GB)...')
pipe = StableDiffusionPipeline.from_pretrained(
    'SG161222/Realistic_Vision_V5.1_noVAE',
    torch_dtype=dtype,
    # use_safetensors=True,  # this repo uses .safetensors
)
# DPM++ 2M Karras: 25 steps = SDXL quality at 35 steps
pipe.scheduler = DPMSolverMultistepScheduler.from_config(
    pipe.scheduler.config,
    use_karras_sigmas=True,
    algorithm_type='dpmsolver++'
)
print('  Realistic Vision ready.')

# ── SD 1.5 IP-Adapter Plus Face ────────────────────────────────────────
# LOAD before any offload (same ordering principle as before)
print('Loading SD 1.5 IP-Adapter Plus Face (~300 MB)...')
pipe.load_ip_adapter(
    'h94/IP-Adapter',
    subfolder='models',
    weight_name='ip-adapter-plus-face_sd15.bin',
)
pipe.set_ip_adapter_scale(0.65)

# Offload + slicing AFTER IP-Adapter
pipe.enable_sequential_cpu_offload()
pipe.vae.enable_slicing()
pipe.vae.enable_tiling()

free_gb = (torch.cuda.get_device_properties(0).total_memory
           - torch.cuda.memory_allocated(0)) / 1024**3
print(f'  Ready. VRAM free: {free_gb:.1f} GB')
print()

print('Generating portrait — 25 steps at 512×768...')
generator = torch.Generator('cpu').manual_seed(SEED)
result = pipe(
    prompt              = PORTRAIT_PROMPT,
    negative_prompt     = PORTRAIT_NEG,
    ip_adapter_image    = face_input,
    width               = 512,
    height              = 768,
    num_inference_steps = 25,
    guidance_scale      = 6.0,
    generator           = generator,
)

result.images[0].save(PORTRAIT_PATH, quality=97)
print(f'Saved: {PORTRAIT_PATH} ({os.path.getsize(PORTRAIT_PATH)//1024} KB)')

del pipe; gc.collect(); torch.cuda.empty_cache()
free_gb = (torch.cuda.get_device_properties(0).total_memory
           - torch.cuda.memory_allocated(0)) / 1024**3
print(f'VRAM freed. Free for Stage 3: {free_gb:.1f} GB')

from IPython.display import display, Image as IPImage
display(IPImage(PORTRAIT_PATH, width=280))
print('STAGE 1 COMPLETE')


---
## 🎬 Cell 6 — Stage 2: Extract Reference Frame
**Time: ~5 seconds**

Uses **ffmpeg** to extract the best pose frame from your reference video.
Tries to extract at the 1-second mark (usually a clear standing pose).
Falls back to the first frame if the video is very short.


In [ ]:
import subprocess, os
from PIL import Image as PILImage
from IPython.display import display, Image as IPImage

FRAME_PATH = '/content/stage2_ref_frame.jpg'

print('=' * 60)
print('STAGE 2 — Reference Frame Extraction')
print('=' * 60)
print(f'  Input : {REFERENCE_VIDEO}')
print(f'  Output: {FRAME_PATH}')
print()

# print video metadata
probe = subprocess.run([
    'ffprobe', '-v', 'error',
    '-select_streams', 'v:0',
    '-show_entries', 'stream=width,height,duration',
    '-of', 'default=noprint_wrappers=1',
    REFERENCE_VIDEO
], capture_output=True, text=True)
print('  Video metadata:')
for line in probe.stdout.strip().split('\n'):
    if line.strip():
        print(f'    {line}')
print()

# Extract frame at 1 second
print('  Extracting frame at t=1.0s...')
subprocess.run([
    'ffmpeg', '-y', '-i', REFERENCE_VIDEO,
    '-ss', '1.0', '-vframes', '1', '-q:v', '2', FRAME_PATH
], capture_output=True)

# Fallback to first frame
if not os.path.exists(FRAME_PATH) or os.path.getsize(FRAME_PATH) < 1000:
    print('  t=1.0s failed — extracting first frame...')
    subprocess.run([
        'ffmpeg', '-y', '-i', REFERENCE_VIDEO,
        '-vframes', '1', '-q:v', '2', FRAME_PATH
    ], capture_output=True)

size_kb = os.path.getsize(FRAME_PATH) / 1024
frame_img = PILImage.open(FRAME_PATH)
fw, fh = frame_img.size
print(f'  Frame saved  : {FRAME_PATH}')
print(f'  Dimensions   : {fw}x{fh}px')
print(f'  File size    : {size_kb:.0f} KB')

print()
print('=' * 60)
print('STAGE 2 COMPLETE')
print('=' * 60)
print('  This frame will be used for pose skeleton extraction in Stage 3:')
display(IPImage(FRAME_PATH, width=320))


---
## 🎭 Cell 7 — Stage 3: Pose and Character Sync
**Time: ~3-4 minutes on T4**

The most important stage. Steps:
1. Extracts the **body pose skeleton** from the reference frame using DWPose
2. Loads **ControlNet OpenPose SDXL** to enforce that exact pose
3. Loads **IP-Adapter Plus Face** to keep the influencer's face identity
4. Generates a new image: same pose, same face, but completely original setting

> **Content Policy:** Prompts use `inspired by` language — NOT `copy` or `same as`.
> Output is transformative original content, not derivative reproduction.


In [ ]:
import os, gc, cv2, torch
from PIL import Image
from diffusers import StableDiffusionControlNetPipeline, ControlNetModel, DPMSolverMultistepScheduler
from controlnet_aux import OpenposeDetector

os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

SYNCED_PATH   = '/content/stage3_synced.jpg'
POSE_DBG_PATH = '/content/stage3_pose_skeleton.jpg'

SYNC_PROMPT = (
    f'RAW photo, (high detailed skin:1.2), {INFLUENCER_CONCEPT}, '
    'inspired by pose composition, unique background, '
    '8k uhd, dslr, professional photography, film grain, Fujifilm XT3'
)
SYNC_NEG = (
    '(deformed iris, deformed pupils, semi-realistic, cgi, 3d, render), '
    'text, worst quality, low quality, blurry, dehydrated, bad anatomy, '
    'bad proportions, extra limbs, disfigured, watermark'
)

print('STAGE 3 — Pose Sync (SD 1.5 ControlNet + IP-Adapter)')
print(f'Peak VRAM target: ~3 GB (vs 9 GB with SDXL ControlNet)')
print()

device = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype  = torch.float16

# Step 3a: Pose skeleton
print('Step 3a: OpenPose skeleton...')
detector = OpenposeDetector.from_pretrained('lllyasviel/ControlNet')
ref_img  = Image.open(FRAME_PATH).convert('RGB')
pose_img = detector(ref_img, include_hand=True, include_face=False)
pose_img.save(POSE_DBG_PATH)
pose_img = pose_img.resize((512, 768), Image.LANCZOS)
print(f'  Skeleton: 512x768')

# Step 3b: Face crop
def crop_face(path, margin=0.12):
    img = cv2.imread(path); h, w = img.shape[:2]
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    det = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
    faces = det.detectMultiScale(gray, 1.05, 3, minSize=(50,50))
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    if len(faces) > 0:
        x, y, fw, fh = faces[0]
        x1,y1=max(0,int(x-margin*fw)),max(0,int(y-margin*fh))
        x2,y2=min(w,int(x+fw+margin*fw)),min(h,int(y+fh+margin*fh))
        return Image.fromarray(rgb[y1:y2,x1:x2])
    s=min(w,h); return Image.fromarray(rgb[(h-s)//2:(h+s)//2,(w-s)//2:(w+s)//2])
face_input = crop_face(PORTRAIT_PATH).resize((224, 224), Image.LANCZOS)
print(f'  Face cropped from portrait')
print()

# Step 3c: SD 1.5 ControlNet OpenPose (~800 MB vs 2.5 GB for SDXL)
print('Step 3c: Loading SD 1.5 ControlNet OpenPose (~800 MB)...')
controlnet = ControlNetModel.from_pretrained(
    'lllyasviel/sd-controlnet-openpose',
    torch_dtype=dtype,
)
pipe = StableDiffusionControlNetPipeline.from_pretrained(
    'SG161222/Realistic_Vision_V5.1_noVAE',
    controlnet=controlnet,
    torch_dtype=dtype,
)
pipe.scheduler = DPMSolverMultistepScheduler.from_config(
    pipe.scheduler.config,
    use_karras_sigmas=True,
    algorithm_type='dpmsolver++'
)

# IP-Adapter before offload
print('Loading SD 1.5 IP-Adapter Plus Face...')
pipe.load_ip_adapter(
    'h94/IP-Adapter',
    subfolder='models',
    weight_name='ip-adapter-plus-face_sd15.bin',
)
pipe.set_ip_adapter_scale(0.65)

# Offload after IP-Adapter
pipe.enable_sequential_cpu_offload()
pipe.vae.enable_slicing()

free_gb = (torch.cuda.get_device_properties(0).total_memory
           - torch.cuda.memory_allocated(0)) / 1024**3
print(f'  Ready. VRAM free: {free_gb:.1f} GB')
print()

print('Generating synced image — 25 steps at 512×768...')
generator = torch.Generator('cpu').manual_seed(SEED)
result = pipe(
    prompt                        = SYNC_PROMPT,
    negative_prompt               = SYNC_NEG,
    image                         = pose_img,
    ip_adapter_image              = face_input,
    width                         = 512,
    height                        = 768,
    num_inference_steps           = 25,
    guidance_scale                = 6.0,
    controlnet_conditioning_scale = float(CONTROLNET_SCALE),
    generator                     = generator,
)

result.images[0].save(SYNCED_PATH, quality=97)
print(f'Saved: {SYNCED_PATH} ({os.path.getsize(SYNCED_PATH)//1024} KB)')

del pipe, controlnet; gc.collect(); torch.cuda.empty_cache()
free_gb = (torch.cuda.get_device_properties(0).total_memory
           - torch.cuda.memory_allocated(0)) / 1024**3
print(f'VRAM freed. Free for Stage 4: {free_gb:.1f} GB')

from IPython.display import display, Image as IPImage
print(); print('STAGE 3 COMPLETE')
display(IPImage(POSE_DBG_PATH, width=180))
display(IPImage(SYNCED_PATH, width=280))


---
## ðŸ“¥ Cell 8 â€” Download Your Influencer Image
Downloads `stage3_synced.jpg` (the final output) to your PC.
Upload it to **Kling AI**, **Pika**, or **Runway** for professional animation.

In [ ]:
from google.colab import files
import os

print('=' * 60)
print('DOWNLOAD FINAL INFLUENCER IMAGE')
print('=' * 60)
print()

SYNCED_PATH   = '/content/stage3_synced.jpg'
PORTRAIT_PATH = '/content/stage1_portrait.jpg'

for label, path in [
    ('Final synced image (Stage 3 â€” upload to Kling)', SYNCED_PATH),
    ('Portrait (Stage 1)', PORTRAIT_PATH),
]:
    if os.path.exists(path):
        kb = os.path.getsize(path) / 1024
        print(f'  Downloading {label}')
        print(f'    {path}  ({kb:.0f} KB)')
        files.download(path)
    else:
        print(f'  Not found: {path}  (run Cells 5-7 first)')
    print()

print('Upload the synced image to Kling AI for high-quality animation.')
print('https://kling.kuaishou.com  (66 free credits/day)')

---
## 📥 Cell 10 — Download Video to Your Computer
Downloads the final video to your PC. It is also saved to Google Drive automatically.


In [ ]:
from google.colab import files
import os

print('=' * 55)
print('DOWNLOAD OUTPUT VIDEO')
print('=' * 55)

if 'VIDEO_PATH' not in dir() or not os.path.exists(VIDEO_PATH):
    print('No video found. Run Cell 8 first to generate the video.')
else:
    mb = os.path.getsize(VIDEO_PATH) / 1024**2
    print(f'  File    : {os.path.basename(VIDEO_PATH)}')
    print(f'  Size    : {mb:.1f} MB')
    print(f'  Location: {VIDEO_PATH}')
    print()
    print('  Starting download — your browser will prompt you to save the file.')
    files.download(VIDEO_PATH)
    print()
    print('  Download started!')
    if OUTPUT_DIR.startswith('/content/drive'):
        print(f'  Also permanently saved in Google Drive at:')
        print(f'  {VIDEO_PATH}')
